In [ ]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[2] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

UNIVERSE = "ALL_SYMBOLS"
START = "2018-05-01"
END = "2026-06-11"
DATASET = "XNAS.ITCH"

df_bars = (
    adapter.scan_data(
        tickers=UNIVERSE,
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d"
    )
    .select(["ts_event", "symbol", "close", "volume"])
    .with_columns(
        pl.col("symbol").cast(pl.Categorical)
    )
)


daily_ret = (
    df_bars
    .filter(
        pl.col("ts_event").dt.date().is_between(pl.date(2024, 1, 1), pl.date(2026, 1, 1))
    )
    .with_columns(
        pl.col("price").pct_change().over("symbol").alias("ret")
    )
    .pivot(
        on="symbol",
        on_columns=UNIVERSE,
        index="date",
        values="ret"
    )
)

daily_ret = daily_ret.collect(engine="streaming")
daily_ret